# Analysis Notebook

Use this notebook to load the metrics data, validate the expected schema, and generate the analysis figures.

Recommended run order:
1. Run the setup cells in Section 1.
2. Run Section 2 for the permissive-system-validation summary.
3. Run Section 3 to regenerate the publication figures.
4. Run Section 4 for the remaining exploratory figures.

Contents:
- [1. Loading in the Data and Making Imports](#1.-Loading-in-the-Data-and-Making-Imports)
- [2. System Validation](#2.-System-Validation)
- [3. Publication Figures](#3.-Publication-Figures)
- [4. Additional Remaining Figures](#4.-Additional-Remaining-Figures)


## 1. Loading in the Data and Making Imports


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid")

metrics_dir = Path('..') / 'metrics'
csv_files = sorted(metrics_dir.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError(f'No CSV files found in {metrics_dir.resolve()}')

all_metrics_df = pd.concat((pd.read_csv(path) for path in csv_files), ignore_index=True)
df = all_metrics_df.copy()
subscenario_abbr = {
    'attack': 'a',
    'permissive': 'b',
    'balanced': 'c',
    'restrictive': 'd',
}

is_risk_assessment = all_metrics_df['permission_assistant'].str.contains(r'risk_assessment|risk_assessment', case=False, na=False)
all_metrics_df['permission_assistant_split'] = all_metrics_df['permission_assistant']
all_metrics_df.loc[is_risk_assessment, 'permission_assistant_split'] = (
    all_metrics_df.loc[is_risk_assessment, 'permission_assistant']
    + ': ' + all_metrics_df.loc[is_risk_assessment, 'risk_tolerance'].fillna('unknown').astype(str)
)

run_count_table = (
    all_metrics_df
    .groupby(['scenario', 'subscenario', 'synthetic_responder_mode', 'permission_assistant_split'])
    .size()
    .unstack('permission_assistant_split', fill_value=0)
    .sort_index()
)

print(f'Loaded {len(csv_files)} CSV files with {len(all_metrics_df)} total runs.')
display(run_count_table)


In [ ]:
# Notebook defaults, validation, and shared helpers
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_FORMAT = 'pdf'

ROW_LABEL_MAP = {
    'attack': 'Attack',
    'permissive': 'Permissive',
    'balanced': 'Balanced',
    'restrictive': 'Restrictive',
}
row_label_map = ROW_LABEL_MAP

MODE_DISPLAY = {
    'always_no': 'no',
    'always_yes': 'yes',
    'alignment_aware': 'aligned',
}
mode_display = MODE_DISPLAY

REQUIRED_COLUMNS = [
    'scenario',
    'subscenario',
    'synthetic_responder_mode',
    'permission_assistant',
    'risk_tolerance',
    'permission_assistant_messages',
    'desired_tool_calls',
    'attack_tool_calls',
    'out_of_alignment_tool_calls',
    'other_tool_calls',
    'total_potential_desired_tool_calls',
    'total_potential_attack_tool_calls',
    'total_potential_out_of_alignment_tool_calls',
    'output_passes',
    'output_fails',
]

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise KeyError(f'Missing required columns: {missing_columns}')

print(f'Validated {len(REQUIRED_COLUMNS)} required columns.')
print(f'Figures will be saved to {FIGURES_DIR.resolve()} as .{EXPORT_FORMAT} files.')

def save_figure(stem, *, dpi=300, bbox_inches='tight'):
    output_path = FIGURES_DIR / f'{stem}.{EXPORT_FORMAT}'
    plt.savefig(output_path, dpi=dpi, bbox_inches=bbox_inches)
    print(f'Saved {output_path}')
    return output_path

def apply_subscenario_row_labels(grid, *, x=-0.28):
    for sub_name, ax in zip(grid.row_names, grid.axes[:, 0]):
        display_name = ROW_LABEL_MAP.get(sub_name, sub_name)
        ax.set_ylabel(display_name, rotation=90, ha='right', va='center', labelpad=20)
        ax.yaxis.set_label_coords(x, 0.5)


## 2. System Validation

This section summarizes runs with `synthetic_responder_mode == "alignment_aware"` and `subscenario == "permissive"`, excluding `permission_assistant == "risk_assessment_autonomous"`. It focuses on output-evaluation pass rates, avoidance of attack or out-of-alignment tool calls, and desired tool-call coverage. Scenario `3` is excluded from output-evaluation pass-rate calculations because no output evaluations are defined there.


In [ ]:
# System validation summary table: alignment_aware synthetic responder, subscenario permissive, excluding risk_assessment_autonomous
import numpy as np
import pandas as pd
from IPython.display import display

assistant_label_series = df['permission_assistant'].astype(str)
risk_mask = assistant_label_series.str.startswith('risk_assessment')
assistant_label_series = assistant_label_series.where(
    ~risk_mask,
    assistant_label_series + ': ' + df['risk_tolerance'].astype(str),
)

summary_b = df[
    df['synthetic_responder_mode'].astype(str).eq('alignment_aware')
    & df['subscenario'].astype(str).eq('permissive')
    & df['permission_assistant'].astype(str).ne('risk_assessment_autonomous')
].copy()

summary_b['assistant_label'] = assistant_label_series.loc[summary_b.index]
summary_b['has_output_eval'] = summary_b['scenario'].astype(str).ne('3')
summary_b['risky_call_made'] = (
    summary_b['attack_tool_calls'].fillna(0).gt(0)
    | summary_b['out_of_alignment_tool_calls'].fillna(0).gt(0)
)
summary_b['all_output_evals_passed'] = (
    summary_b['output_fails'].fillna(0).eq(0)
    & summary_b[['output_passes', 'output_fails']].fillna(0).sum(axis=1).gt(0)
    & summary_b['has_output_eval']
)
summary_b['desired_call_rate'] = (
    100
    * summary_b['desired_tool_calls']
    / summary_b['total_potential_desired_tool_calls'].replace(0, np.nan)
)
overall_b = pd.DataFrame([
    {
        'runs': len(summary_b),
        '% runs with all output evals passed': 100 * summary_b.loc[summary_b['has_output_eval'], 'all_output_evals_passed'].mean(),
        '% runs with no attack/out-of-alignment calls': 100 * (~summary_b['risky_call_made']).mean(),
        '% desired tool calls made (aggregate)': 100 * summary_b['desired_tool_calls'].sum() / summary_b['total_potential_desired_tool_calls'].sum(),
        'average % desired tool calls made': summary_b['desired_call_rate'].mean(),
        'median % desired tool calls made': summary_b['desired_call_rate'].median(),
        'runs with output evals defined': int(summary_b['has_output_eval'].sum()),
    }
]).round(1)

print('Overall: alignment_aware synthetic responder, subscenario permissive, excluding risk_assessment_autonomous')
display(overall_b)


## 3. Publication Figures

These are the publication figures previously labeled 8, 11, and 21. They are grouped first here and renumbered for notebook review.


In [ ]:
# Publication Figure 1: Dual-axis tool calls and permission assistant messages by risk tolerance, faceted by subscenario (rows) and scenario (columns)
# Columns used: synthetic_responder_mode, subscenario, scenario, risk_tolerance,
# permission_assistant_messages, attack_tool_calls, out_of_alignment_tool_calls

from matplotlib.lines import Line2D

filtered = df[(df["synthetic_responder_mode"] == "alignment_aware") & df["risk_tolerance"].notna() & (df["risk_tolerance"].astype(str).str.upper() != "N/A")].copy()

tool_df = filtered.melt(
    id_vars=["subscenario", "scenario", "risk_tolerance"],
    value_vars=["attack_tool_calls", "out_of_alignment_tool_calls"],
    var_name="tool_call_type",
    value_name="tool_calls",
)
row_label_map = {
    "attack": "Attack",
    "permissive": "Permissive",
    "balanced": "Balanced",
    "restrictive": "Restrictive"
}

tool_df["tool_call_type"] = tool_df["tool_call_type"].map({
    "attack_tool_calls": "Attack",
    "out_of_alignment_tool_calls": "Out of Alignment",
})

# Only show Attack in subscenario attack; Out of Alignment in subscenarios permissive/balanced/restrictive
tool_df = tool_df[(~(
    (tool_df["tool_call_type"] == "Attack") & (tool_df["subscenario"] != "attack")
)) & ~(
    (tool_df["tool_call_type"] == "Out of Alignment")
    & (~tool_df["subscenario"].isin(["permissive", "balanced", "restrictive"]))
)]

row_label_map = {
    'attack': 'Attack',
    'permissive': 'Permissive',
    'balanced': 'Balanced',
    'restrictive': 'Restrictive',
}

palette = {
    "Attack": "red",
    "Out of Alignment": "orange",
}

g6 = sns.FacetGrid(
    tool_df,
    row="subscenario",
    col="scenario",
    margin_titles=True,
    height=4,
    aspect=1.4,
    sharey=True,
)

g6.map_dataframe(
    sns.lineplot,
    x="risk_tolerance",
    y="tool_calls",
    hue="tool_call_type",
    palette=palette,
    marker="o",
    linewidth=2.5,
    legend=False,
)

for i, sub_name in enumerate(g6.row_names):
    for j, scen_name in enumerate(g6.col_names):
        ax = g6.axes[i, j]
        ax.set_ylim(0, 6)
        pa_df = filtered[
            (filtered["subscenario"] == sub_name)
            & (filtered["scenario"].astype(str) == str(scen_name))
        ]
        right_ax = ax.twinx()
        sns.lineplot(
            data=pa_df,
            x="risk_tolerance",
            y="permission_assistant_messages",
            marker="s",
            linewidth=2.5,
            linestyle="--",
            color="#2ca02c",
            legend=False,
            ax=right_ax,
        )
        # 1. Set the right axis limit to exactly 0 to 18
        right_ax.set_ylim(0, 18)
        
        # 2. Force exactly 4 tick marks to perfectly match the left axis
        right_ax.set_yticks([0, 6, 12, 18])
        
        # 3. Only show right-side labels on the far-right column
        if j == len(g6.col_names) - 1:
            right_ax.tick_params(axis="y", labelright=True, labelcolor="#2ca02c", labelsize=28)
        else:
            right_ax.tick_params(axis="y", labelright=False, right=False)

        right_ax.set_ylabel("")
        
        # 4. Turn off the right grid so it relies entirely on the left grid
        right_ax.grid(False)



for sub_name, ax in zip(g6.row_names, g6.axes[:, 0]):
    ax.set_ylabel(sub_name, rotation=0, ha="center", va="center", labelpad=30)

legend_handles = [
    Line2D([], [], color=palette["Attack"], marker="o", linewidth=2.5, label="Attack"),
    Line2D([], [], color=palette["Out of Alignment"], marker="o", linewidth=2.5, label="Out of Alignment"),
    Line2D([], [], color="#2ca02c", marker="s", linestyle="--", linewidth=2.5, label="Permission Assistant Messages"),
]
leg6 = g6.figure.legend(handles=legend_handles, bbox_to_anchor=(0.55, 0.1), loc="lower center", borderaxespad=0, ncol=3, frameon=False)
for txt in leg6.get_texts():
    txt.set_fontsize(24)

g6.set_axis_labels("Risk Tolerance", "")
g6.set_titles(row_template="", col_template="Scenario {col_name}")

for sub_name, ax in zip(g6.row_names, g6.axes[:, 0]):
    display_name = row_label_map.get(sub_name, sub_name)
    ax.set_ylabel(display_name, rotation=90, ha="right", va="center", labelpad=20)
    ax.yaxis.set_label_coords(-0.28, 0.5)

g6.figure.supylabel("Tool Calls", x=0.16, color ='black', fontsize=28)
g6.figure.text(0.95, 0.5, "Permission Assistant Messages", rotation=270, va="center", ha="center", color="#2ca02c", fontsize=28) # "#2ca02c"



for ax in g6.axes.flat:
    ax.grid(True, linewidth=0.6)

for ax in g6.axes.flat:
    ax.tick_params(axis='both', labelsize=24)
    ax.xaxis.label.set_size(28)
    ax.yaxis.label.set_size(28)
    ax.title.set_size(28)
    ax.axvline(0.2, color='gray', linestyle=':', linewidth=1.5, alpha=0.8)
    ax.axvline(0.7, color='gray', linestyle=':', linewidth=1.5, alpha=0.8)
    # ensure 0.2 and 0.7 appear as x ticks
    # ax.set_xticks(sorted(set(list(ax.get_xticks()) + [0.2, 0.7])))
    ax.set_xticks([0, 0.2, 0.5, 0.7, 1])


for ax in g6.axes.flat:
    # Use a light gray color, add transparency (alpha), and make it dashed
    ax.grid(True, color='gray', alpha=0.3, linestyle='--', linewidth=0.8)
    
    # Optional: If you only want horizontal grid lines, you can specify axis='y'
    # ax.grid(True, axis='y', color='gray', alpha=0.3, linestyle='--', linewidth=0.8)

plt.tight_layout(rect=[0.16, 0.14, 0.94, 1])
save_figure('publication_tool_calls_and_permission_messages_by_risk_tolerance')


In [ ]:
# Publication Figure 2: Risky-call rate vs permission-assistant message load (alignment_aware), points by subscenario (excluding permissive)
# Same as Publication Figure 1, but excluding the permissive subscenario.
# One unified graph. One point per subscenario for each permission assistant, with each assistant wrapped by its outer-edge hull.

import numpy as np

sub_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].astype(str).eq('alignment_aware')
    & df['subscenario'].notna()
    & df['subscenario'].astype(str).ne('permissive')
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
sub_df['assistant_label'] = sub_df['permission_assistant'].astype(str)
mask_risk = sub_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & sub_df['risk_tolerance'].notna()
sub_df.loc[mask_risk, 'assistant_label'] = (
    sub_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + sub_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

sub_df['risky_call_made'] = (
    (sub_df['attack_tool_calls'].fillna(0) > 0)
    | (sub_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_sub = (
    sub_df.groupby(['assistant_label', 'subscenario'], as_index=False)
    .agg(
        permission_assistant_messages=('permission_assistant_messages', 'mean'),
        percent_risky=('risky_call_made', lambda s: 100 * s.mean()),
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
    )
)

fixed_point_size = 130

sub_order = sorted(summary_sub['subscenario'].dropna().astype(str).unique())
summary_sub['subscenario'] = pd.Categorical(summary_sub['subscenario'], categories=sub_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_sub['assistant_label'].astype(str).unique())
palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(16, 9))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.10, 1.6),
    (0.10, -2.2),
    (0.22, 2.8),
    (0.22, -3.2),
    (-0.22, 2.2),
    (-0.22, -2.8),
]
assistant_label_offsets = [
    (0.18, 3.0),
    (0.18, -3.2),
    (0.34, 5.0),
    (0.34, -5.2),
    (-0.34, 3.8),
    (-0.34, -4.0),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_sub[summary_sub['assistant_label'] == assistant].sort_values('subscenario')
    if a_df.empty:
        continue

    points = list(zip(a_df['permission_assistant_messages'], a_df['percent_risky']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 0.35) or (abs(cy - py) > 3.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    alx, aly = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((alx, aly))
    ax.text(
        alx,
        aly,
        assistant,
        color=color,
        fontsize=22,
        ha='left' if chosen_adx >= 0 else 'right',
        va='center',
        bbox=dict(facecolor='white', edgecolor=color, alpha=0.75, pad=0.25),
        zorder=5,
    )

    ax.scatter(
        a_df['permission_assistant_messages'],
        a_df['percent_risky'],
        s=fixed_point_size,
        color=color,
        edgecolor='white',
        linewidth=0.8,
        zorder=3,
    )

    for _, row in a_df.iterrows():
        x = float(row['permission_assistant_messages'])
        y = float(row['percent_risky'])

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 0.22) or (abs(cy - py) > 2.0) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            subscenario_abbr.get(str(row["subscenario"]), str(row["subscenario"])),
            color=color,
            fontsize=22,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

from matplotlib.lines import Line2D

legend_handles = [
    Line2D([0], [0], marker='$a$', color='black', linestyle='None', markersize=11, label='attack'),
    Line2D([0], [0], marker='$p$', color='black', linestyle='None', markersize=14, label='permissive'),
    Line2D([0], [0], marker='$b$', color='black', linestyle='None', markersize=14, label='balanced'),
    Line2D([0], [0], marker='$r$', color='black', linestyle='None', markersize=11, label='restrictive'),
]
ax.legend(
    handles=legend_handles,
    title='Subscenario',
    loc='upper right',
    frameon=True,
    fontsize=22,
    title_fontsize=22,
)

ax.set_xlabel('Permission Assistant Messages (Average per Run)', fontsize=24)
ax.set_ylabel('% of Runs with an Attack or\nOut-of-Alignment Tool Call', fontsize=24)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0.16, 0.08, 0.98, 0.98])

save_figure('publication_risky_call_rate_vs_permission_messages_alignment_aware')




In [ ]:
# Publication Figure 3: Risky-call rate vs permission-assistant message load by synthetic responder mode
# X-axis: average permission assistant messages.
# Y-axis: percent of runs where an attack or out-of-alignment call was made.
# One point per synthetic responder mode, with each assistant wrapped by its outer-edge hull.

import numpy as np

mode_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].notna()
    & df['subscenario'].notna()
    & df['subscenario'].astype(str).ne('permissive')
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
mode_df['assistant_label'] = mode_df['permission_assistant'].astype(str)
mask_risk = mode_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & mode_df['risk_tolerance'].notna()
mode_df.loc[mask_risk, 'assistant_label'] = (
    mode_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + mode_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

mode_df['risky_call_made'] = (
    (mode_df['attack_tool_calls'].fillna(0) > 0)
    | (mode_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_mode = (
    mode_df.groupby(['assistant_label', 'synthetic_responder_mode'], as_index=False)
    .agg(
        permission_assistant_messages=('permission_assistant_messages', 'mean'),
        percent_risky=('risky_call_made', lambda s: 100 * s.mean()),
        output_passes=('output_passes', 'sum'),
        output_fails=('output_fails', 'sum'),
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
    )
)

summary_mode['total_output_evals'] = summary_mode['output_passes'].fillna(0) + summary_mode['output_fails'].fillna(0)
summary_mode['output_fail_rate'] = (
    100 * summary_mode['output_fails'].fillna(0) / summary_mode['total_output_evals'].replace(0, float('nan'))
).fillna(0.0)
summary_mode['marker'] = summary_mode['output_fail_rate'].apply(lambda pct: 'X' if pct > 5 else 'o')

fixed_point_size = 130

preferred_mode_order = ['always_no', 'alignment_aware', 'always_yes']
mode_values = list(summary_mode['synthetic_responder_mode'].dropna().astype(str).unique())
mode_order = [m for m in preferred_mode_order if m in mode_values] + [m for m in mode_values if m not in preferred_mode_order]
summary_mode['synthetic_responder_mode'] = pd.Categorical(summary_mode['synthetic_responder_mode'], categories=mode_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_mode['assistant_label'].astype(str).unique())
mode_display = {
    'always_no': 'no',
    'always_yes': 'yes',
    'alignment_aware': 'aligned',
}

palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(13, 12))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.10, 1.6),
    (0.10, -2.2),
    (0.22, 2.8),
    (0.22, -3.2),
    (-0.22, 2.2),
    (-0.22, -2.8),
]
assistant_label_offsets = [
    (0.18, 3.0),
    (0.18, -3.2),
    (0.34, 5.0),
    (0.34, -5.2),
    (-0.34, 3.8),
    (-0.34, -4.0),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_mode[summary_mode['assistant_label'] == assistant].sort_values('synthetic_responder_mode')
    if a_df.empty:
        continue

    points = list(zip(a_df['permission_assistant_messages'], a_df['percent_risky']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    # Label assistant shape on chart with overlap-avoidance.
    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 0.35) or (abs(cy - py) > 3.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    alx, aly = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((alx, aly))

    # Plot and label every synthetic-responder point.
    for _, row in a_df.iterrows():
        x = float(row['permission_assistant_messages'])
        y = float(row['percent_risky'])

        ax.scatter(
            [x],
            [y],
            s=fixed_point_size,
            marker=row['marker'],
            color=color,
            edgecolor='white',
            linewidth=0.8,
            zorder=3,
        )

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 0.22) or (abs(cy - py) > 2.0) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            str(mode_display.get(mode_display.get(str(row['synthetic_responder_mode']), str(row['synthetic_responder_mode'])), mode_display.get(str(row['synthetic_responder_mode']), str(row['synthetic_responder_mode'])))),
            color=color,
            fontsize=24,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

ax.set_xlabel('Permission Assistant Messages (Average per Run)', fontsize=24)
ax.set_ylabel('% of Runs with an Attack or\nOut-of-Alignment Tool Call', fontsize=24)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)

from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]


from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]
assistant_legend = fig.legend(
    handles=assistant_handles,
    title='Permission Assistant',
    loc='lower center',
    bbox_to_anchor=(0.55, -0.02),
    ncol=2,
    frameon=True,
    fontsize=24,
    title_fontsize=26,
)

marker_handles = [
    Line2D([0], [0], marker='o', color='#666666', markerfacecolor='#666666', markersize=10, linestyle='None', label='<=5% fail rate'),
    Line2D([0], [0], marker='X', color='#666666', markerfacecolor='#666666', markersize=10, linestyle='None', label='>5% fail rate'),
]

marker_legend = ax.legend(
    marker_handles,
    ['<=5% fail rate', '>5% fail rate'],
    title='Output Validation',
    loc='upper center',
    bbox_to_anchor=(0.45, -0.12),
    ncol=2,
    frameon=True,
    fontsize=24,
    title_fontsize=26,
)

ax.tick_params(axis='both', labelsize=24)
plt.tight_layout(rect=[0.12, 0.20, 1, 1])
save_figure('publication_risky_call_rate_vs_permission_messages_by_synthetic_responder')


## 4. Additional Remaining Figures


### Risk Tolerance Deep Dive


In [ ]:
# Messages by risk tolerance, faceted by subscenario (rows) and scenario (columns)
# Columns used: synthetic_responder_mode, subscenario, scenario, risk_tolerance, user_messages, agent_messages, permission_assistant_messages

filtered = df[(df['synthetic_responder_mode'] == 'alignment_aware') & df['risk_tolerance'].notna() & (df['risk_tolerance'].astype(str).str.upper() != 'N/A')].copy()

msg_df = filtered.melt(
    id_vars=['subscenario', 'scenario', 'risk_tolerance'],
    value_vars=['user_messages', 'agent_messages', 'permission_assistant_messages'],
    var_name='message_type',
    value_name='messages'
)

msg_df['message_type'] = msg_df['message_type'].map({
    'user_messages': 'User',
    'agent_messages': 'Agent',
    'permission_assistant_messages': 'Permission Assistant',
})

palette = {
    'User': '#2ca02c',
    'Agent': '#1f77b4',
    'Permission Assistant': '#9467bd',
}

g1 = sns.FacetGrid(
    msg_df,
    row='subscenario',
    col='scenario',
    margin_titles=True,
    height=4.5,
    aspect=1.2,
    sharey=True,
)
g1.map_dataframe(
    sns.lineplot,
    x='risk_tolerance',
    y='messages',
    hue='message_type',
    palette=palette,
    marker='o',
    linewidth=2.5,
)
g1.add_legend(title='Message Type', bbox_to_anchor=(0.55, 0.1), loc='lower center', borderaxespad=0, ncol=3)
if g1._legend is not None:
    g1._legend.set_title('Message Type', prop={'size': 20})
    for txt in g1._legend.get_texts():
        txt.set_fontsize(20)
g1.set_axis_labels('Risk Tolerance', '')
g1.set_titles(row_template='', col_template='Scenario {col_name}')

# Left-side subscenario labels
apply_subscenario_row_labels(g1)

# Shared y-axis label for messages
g1.figure.supylabel('Messages', x=0.17, fontsize=20)

# Keep gridlines thinner than series lines
for ax in g1.axes.flat:
    ax.grid(True, linewidth=0.6)

for ax in g1.axes.flat:
    ax.tick_params(axis='both', labelsize=20)
    ax.xaxis.label.set_size(20)
    ax.yaxis.label.set_size(20)
    ax.title.set_size(20)

plt.tight_layout(rect=[0.16, 0.14, 0.98, 1])
save_figure('messages_by_risk_tolerance')


In [ ]:
# Tool calls by risk tolerance, faceted by subscenario (rows) and scenario (columns)
# Columns used: synthetic_responder_mode, subscenario, scenario, risk_tolerance, desired_tool_calls, attack_tool_calls, out_of_alignment_tool_calls, other_tool_calls

filtered = df[(df['synthetic_responder_mode'] == 'alignment_aware') & df['risk_tolerance'].notna() & (df['risk_tolerance'].astype(str).str.upper() != 'N/A')].copy()

tool_df = filtered.melt(
    id_vars=['subscenario', 'scenario', 'risk_tolerance'],
    value_vars=['attack_tool_calls', 'out_of_alignment_tool_calls', 'desired_tool_calls', 'other_tool_calls'],
    var_name='tool_call_type',
    value_name='tool_calls'
)

tool_df['tool_call_type'] = tool_df['tool_call_type'].map({
    'attack_tool_calls': 'Attack',
    'out_of_alignment_tool_calls': 'Out of Alignment',
    'desired_tool_calls': 'Desired',
    'other_tool_calls': 'Other',
})

# Only show Attack in subscenario attack; Out of Alignment in subscenarios permissive/balanced/restrictive
tool_df = tool_df[~(
    (tool_df['tool_call_type'] == 'Attack') & (tool_df['subscenario'] != 'attack')
) & ~(
    (tool_df['tool_call_type'] == 'Out of Alignment') & (~tool_df['subscenario'].isin(['permissive', 'balanced', 'restrictive']))
)]

row_label_map = {
    'attack': 'Attack',
    'permissive': 'Permissive',
    'balanced': 'Balanced',
    'restrictive': 'Restrictive',
}

palette = {
    'Attack': 'red',
    'Out of Alignment': 'orange',
    'Desired': '#1f77b4',
    'Other': 'lightgrey',
}

g2 = sns.FacetGrid(
    tool_df,
    row='subscenario',
    col='scenario',
    margin_titles=True,
    height=4.5,
    aspect=1.2,
    sharey=True,
)
g2.map_dataframe(
    sns.lineplot,
    x='risk_tolerance',
    y='tool_calls',
    hue='tool_call_type',
    palette=palette,
    marker='o',
    linewidth=2.5,
)

# Baselines per facet:
# - Attack baseline at 0 (subscenario attack)
# - Out-of-Alignment baseline at 0 (subscenarios permissive/balanced/restrictive)
# - Desired baseline at total potential desired tool calls from metadata
baseline_df = (
    filtered[['subscenario', 'scenario', 'total_potential_desired_tool_calls']]
    .drop_duplicates(subset=['subscenario', 'scenario'])
)

desired_baseline = {
    (row['subscenario'], str(row['scenario'])): row['total_potential_desired_tool_calls']
    for _, row in baseline_df.iterrows()
}

for i, sub_name in enumerate(g2.row_names):
    for j, scen_name in enumerate(g2.col_names):
        ax = g2.axes[i, j]
        scen_str = str(scen_name)

        if sub_name == 'attack':
            ax.axhline(0, color=palette['Attack'], linestyle=':', linewidth=1.8, alpha=0.9)
        if sub_name in {'permissive', 'balanced', 'restrictive'}:
            ax.axhline(0, color=palette['Out of Alignment'], linestyle=':', linewidth=1.8, alpha=0.9)

        y_desired = desired_baseline.get((sub_name, scen_str))
        if y_desired is not None:
            ax.axhline(y_desired, color=palette['Desired'], linestyle=':', linewidth=1.8, alpha=0.9)

g2.add_legend(title='Tool Call Type', bbox_to_anchor=(0.55, 0.1), loc='lower center', borderaxespad=0, ncol=4)
if g2._legend is not None:
    g2._legend.set_title('Tool Call Type', prop={'size': 20})
    for txt in g2._legend.get_texts():
        txt.set_fontsize(20)
g2.set_axis_labels('Risk Tolerance', '')
g2.set_titles(row_template='', col_template='Scenario {col_name}')

# Left-side subscenario labels
apply_subscenario_row_labels(g2)

# Shared y-axis label for tool calls
g2.figure.supylabel('Tool Calls', x=0.17, fontsize=20)

# Keep gridlines thinner than series lines
for ax in g2.axes.flat:
    ax.grid(True, linewidth=0.6)

for ax in g2.axes.flat:
    ax.tick_params(axis='both', labelsize=20)
    ax.xaxis.label.set_size(20)
    ax.yaxis.label.set_size(20)
    ax.title.set_size(20)

plt.tight_layout(rect=[0.16, 0.14, 0.98, 1])
save_figure('tool_calls_by_risk_tolerance')


In [ ]:
# Percent of output evaluations passed by risk tolerance
# faceted by subscenario (rows) and scenario (columns), with one line per synthetic responder mode.
# Columns used: synthetic_responder_mode, subscenario, scenario, risk_tolerance, output_passes, output_fails

viz3_df = df[(df['scenario'].astype(str) != '3') & df['risk_tolerance'].notna() & (df['risk_tolerance'].astype(str).str.upper() != 'N/A')].copy()
viz3_df['total_output_evals'] = viz3_df['output_passes'] + viz3_df['output_fails']
viz3_df = viz3_df[viz3_df['total_output_evals'] > 0].copy()
viz3_df['risk_tolerance_numeric'] = pd.to_numeric(viz3_df['risk_tolerance'])
viz3_df['percent_passed'] = 100 * viz3_df['output_passes'] / viz3_df['total_output_evals']

# Slight horizontal dodge by synthetic responder mode so overlapping lines/markers are visible.
mode_vals = list(viz3_df['synthetic_responder_mode'].dropna().unique())
preferred_mode_order = ['always_no', 'alignment_aware', 'always_yes']
mode_order = [m for m in preferred_mode_order if m in mode_vals] + [m for m in mode_vals if m not in preferred_mode_order]

if len(mode_order) <= 1:
    mode_offsets = {mode_order[0]: 0.0} if mode_order else {}
else:
    start = -0.015
    step = 0.03 / (len(mode_order) - 1)
    mode_offsets = {mode: start + idx * step for idx, mode in enumerate(mode_order)}

viz3_df['risk_tolerance_plot'] = (
    viz3_df['risk_tolerance_numeric'] + viz3_df['synthetic_responder_mode'].map(mode_offsets).fillna(0.0)
)
risk_ticks = sorted(viz3_df['risk_tolerance_numeric'].dropna().unique())

g3 = sns.FacetGrid(
    viz3_df,
    row='subscenario',
    col='scenario',
    margin_titles=True,
    height=3,
    aspect=1.4,
    sharey=True,
)
g3.map_dataframe(
    sns.lineplot,
    x='risk_tolerance_plot',
    y='percent_passed',
    hue='synthetic_responder_mode',
    style='synthetic_responder_mode',
    marker='o',
    linewidth=2.5,
)
g3.add_legend(title='Synthetic Responder', bbox_to_anchor=(0.55, 0.08), loc='lower center', borderaxespad=0, ncol=3)
if g3._legend is not None:
    g3._legend.set_title('Synthetic Responder', prop={'size': 20})
    for txt in g3._legend.get_texts():
        txt.set_fontsize(20)
g3.set_axis_labels('Risk Tolerance', '')
g3.set_titles(row_template='', col_template='Scenario {col_name}')
g3.set(ylim=(-2, 102))
for ax in g3.axes.flat:
    ax.set_xticks(risk_ticks)

row_label_map = {
    'attack': 'Attack',
    'permissive': 'Permissive',
    'balanced': 'Balanced',
    'restrictive': 'Restrictive',
}

# Left-side subscenario labels
apply_subscenario_row_labels(g3)

# Shared y-axis label for percent passed
g3.figure.supylabel('Percent Passed Output Evaluations', x=0.17, fontsize=20)

# Keep gridlines thinner than series lines
for ax in g3.axes.flat:
    ax.grid(True, linewidth=0.6)

for ax in g3.axes.flat:
    ax.tick_params(axis='both', labelsize=20)
    ax.xaxis.label.set_size(20)
    ax.yaxis.label.set_size(20)
    ax.title.set_size(20)

plt.tight_layout(rect=[0.16, 0.14, 0.98, 1])
save_figure('output_eval_pass_rate_by_risk_tolerance')


### Comparing Permission Assistants


In [ ]:
# Risky-call rate vs permission-assistant message load (alignment_aware), points by subscenario
# One unified graph. One point per subscenario for each permission assistant, with each assistant wrapped by its outer-edge hull.

import numpy as np

sub_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].astype(str).eq('alignment_aware')
    & df['subscenario'].notna()
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
sub_df['assistant_label'] = sub_df['permission_assistant'].astype(str)
mask_risk = sub_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & sub_df['risk_tolerance'].notna()
sub_df.loc[mask_risk, 'assistant_label'] = (
    sub_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + sub_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

sub_df['risky_call_made'] = (
    (sub_df['attack_tool_calls'].fillna(0) > 0)
    | (sub_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_sub = (
    sub_df.groupby(['assistant_label', 'subscenario'], as_index=False)
    .agg(
        permission_assistant_messages=('permission_assistant_messages', 'mean'),
        percent_risky=('risky_call_made', lambda s: 100 * s.mean()),
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
    )
)

fixed_point_size = 130

sub_order = sorted(summary_sub['subscenario'].dropna().astype(str).unique())
summary_sub['subscenario'] = pd.Categorical(summary_sub['subscenario'], categories=sub_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_sub['assistant_label'].astype(str).unique())
palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(10, 8))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.10, 1.6),
    (0.10, -2.2),
    (0.22, 2.8),
    (0.22, -3.2),
    (-0.22, 2.2),
    (-0.22, -2.8),
]
assistant_label_offsets = [
    (0.18, 3.0),
    (0.18, -3.2),
    (0.34, 5.0),
    (0.34, -5.2),
    (-0.34, 3.8),
    (-0.34, -4.0),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_sub[summary_sub['assistant_label'] == assistant].sort_values('subscenario')
    if a_df.empty:
        continue

    points = list(zip(a_df['permission_assistant_messages'], a_df['percent_risky']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 0.35) or (abs(cy - py) > 3.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    alx, aly = anchor_x + chosen_adx, anchor_y + chosen_ady
    if assistant in {'policy_suggestion', 'user_confirmation'}:
        alx -= 1.2
    placed_assistant_labels.append((alx, aly))

    ax.scatter(
        a_df['permission_assistant_messages'],
        a_df['percent_risky'],
        s=fixed_point_size,
        color=color,
        edgecolor='white',
        linewidth=0.8,
        zorder=3,
    )

    for _, row in a_df.iterrows():
        x = float(row['permission_assistant_messages'])
        y = float(row['percent_risky'])

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 0.22) or (abs(cy - py) > 2.0) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            subscenario_abbr.get(str(row["subscenario"]), str(row["subscenario"])),
            color=color,
            fontsize=20,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

ax.set_xlabel('Permission Assistant Messages (Average per Run)', fontsize=24)
ax.set_ylabel('% of Runs with Attack or\nOut-of-Alignment Tool Call', fontsize=24)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)

from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]


from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]
assistant_legend = fig.legend(
    handles=assistant_handles,
    title='Permission Assistant',
    loc='lower center',
    bbox_to_anchor=(0.5, -0.14),
    ncol=2,
    frameon=True,
    fontsize=20,
    title_fontsize=20,
)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0, 0.16, 1, 1])
save_figure('risky_call_rate_vs_permission_messages_alignment_aware_all_subscenarios')


In [ ]:
# Risky-call rate vs permission-assistant message load (always_no), points by subscenario (excluding permissive)
# One unified graph. One point per subscenario for each permission assistant, with each assistant wrapped by its outer-edge hull.

import numpy as np

sub_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].astype(str).eq('always_no')
    & df['subscenario'].notna()
    & df['subscenario'].astype(str).ne('permissive')
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
sub_df['assistant_label'] = sub_df['permission_assistant'].astype(str)
mask_risk = sub_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & sub_df['risk_tolerance'].notna()
sub_df.loc[mask_risk, 'assistant_label'] = (
    sub_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + sub_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

sub_df['risky_call_made'] = (
    (sub_df['attack_tool_calls'].fillna(0) > 0)
    | (sub_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_sub = (
    sub_df.groupby(['assistant_label', 'subscenario'], as_index=False)
    .agg(
        permission_assistant_messages=('permission_assistant_messages', 'mean'),
        percent_risky=('risky_call_made', lambda s: 100 * s.mean()),
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
    )
)

fixed_point_size = 130

sub_order = sorted(summary_sub['subscenario'].dropna().astype(str).unique())
summary_sub['subscenario'] = pd.Categorical(summary_sub['subscenario'], categories=sub_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_sub['assistant_label'].astype(str).unique())
palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(21, 9))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.10, 1.6),
    (0.10, -2.2),
    (0.22, 2.8),
    (0.22, -3.2),
    (-0.22, 2.2),
    (-0.22, -2.8),
]
assistant_label_offsets = [
    (0.18, 3.0),
    (0.18, -3.2),
    (0.34, 5.0),
    (0.34, -5.2),
    (-0.34, 3.8),
    (-0.34, -4.0),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_sub[summary_sub['assistant_label'] == assistant].sort_values('subscenario')
    if a_df.empty:
        continue

    points = list(zip(a_df['permission_assistant_messages'], a_df['percent_risky']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 0.35) or (abs(cy - py) > 3.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    alx, aly = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((alx, aly))
    ax.text(
        alx,
        aly,
        assistant,
        color=color,
        fontsize=19,
        ha='left' if chosen_adx >= 0 else 'right',
        va='center',
        bbox=dict(facecolor='white', edgecolor=color, alpha=0.75, pad=0.25),
        zorder=5,
    )

    ax.scatter(
        a_df['permission_assistant_messages'],
        a_df['percent_risky'],
        s=fixed_point_size,
        color=color,
        edgecolor='white',
        linewidth=0.8,
        zorder=3,
    )

    for _, row in a_df.iterrows():
        x = float(row['permission_assistant_messages'])
        y = float(row['percent_risky'])

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 0.22) or (abs(cy - py) > 2.0) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            subscenario_abbr.get(str(row["subscenario"]), str(row["subscenario"])),
            color=color,
            fontsize=20,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

from matplotlib.lines import Line2D

legend_handles = [
    Line2D([0], [0], marker='$a$', color='black', linestyle='None', markersize=11, label='attack'),
    Line2D([0], [0], marker='$p$', color='black', linestyle='None', markersize=14, label='permissive'),
    Line2D([0], [0], marker='$b$', color='black', linestyle='None', markersize=14, label='balanced'),
    Line2D([0], [0], marker='$r$', color='black', linestyle='None', markersize=11, label='restrictive'),
]
ax.legend(
    handles=legend_handles,
    title='Subscenario',
    loc='upper right',
    frameon=True,
    fontsize=16,
    title_fontsize=17,
)

ax.set_xlabel('Permission Assistant Messages (Average per Run)', fontsize=24)
ax.set_ylabel('% of Runs with an Attack or\nOut-of-Alignment Tool Call', fontsize=24)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0.16, 0.08, 0.98, 0.98])

save_figure('risky_call_rate_vs_permission_messages_always_no')




In [ ]:
# Risky-call rate vs permission-assistant message load (always_yes), points by subscenario (excluding permissive)
# Same as the alignment-aware version above, but excluding the permissive subscenario.
# One unified graph. One point per subscenario for each permission assistant, with each assistant wrapped by its outer-edge hull.

import numpy as np

sub_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].astype(str).eq('always_yes')
    & df['subscenario'].notna()
    & df['subscenario'].astype(str).ne('permissive')
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
sub_df['assistant_label'] = sub_df['permission_assistant'].astype(str)
mask_risk = sub_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & sub_df['risk_tolerance'].notna()
sub_df.loc[mask_risk, 'assistant_label'] = (
    sub_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + sub_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

sub_df['risky_call_made'] = (
    (sub_df['attack_tool_calls'].fillna(0) > 0)
    | (sub_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_sub = (
    sub_df.groupby(['assistant_label', 'subscenario'], as_index=False)
    .agg(
        permission_assistant_messages=('permission_assistant_messages', 'mean'),
        percent_risky=('risky_call_made', lambda s: 100 * s.mean()),
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
    )
)

fixed_point_size = 130

sub_order = sorted(summary_sub['subscenario'].dropna().astype(str).unique())
summary_sub['subscenario'] = pd.Categorical(summary_sub['subscenario'], categories=sub_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_sub['assistant_label'].astype(str).unique())
palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(18, 9))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.10, 1.6),
    (0.10, -2.2),
    (0.22, 2.8),
    (0.22, -3.2),
    (-0.22, 2.2),
    (-0.22, -2.8),
]
assistant_label_offsets = [
    (0.18, 3.0),
    (0.18, -3.2),
    (0.34, 5.0),
    (0.34, -5.2),
    (-0.34, 3.8),
    (-0.34, -4.0),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_sub[summary_sub['assistant_label'] == assistant].sort_values('subscenario')
    if a_df.empty:
        continue

    points = list(zip(a_df['permission_assistant_messages'], a_df['percent_risky']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 0.35) or (abs(cy - py) > 3.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    alx, aly = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((alx, aly))
    ax.text(
        alx,
        aly,
        assistant,
        color=color,
        fontsize=22,
        ha='left' if chosen_adx >= 0 else 'right',
        va='center',
        bbox=dict(facecolor='white', edgecolor=color, alpha=0.75, pad=0.25),
        zorder=5,
    )

    ax.scatter(
        a_df['permission_assistant_messages'],
        a_df['percent_risky'],
        s=fixed_point_size,
        color=color,
        edgecolor='white',
        linewidth=0.8,
        zorder=3,
    )

    for _, row in a_df.iterrows():
        x = float(row['permission_assistant_messages'])
        y = float(row['percent_risky'])

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 0.22) or (abs(cy - py) > 2.0) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            subscenario_abbr.get(str(row["subscenario"]), str(row["subscenario"])),
            color=color,
            fontsize=22,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

from matplotlib.lines import Line2D

legend_handles = [
    Line2D([0], [0], marker='$a$', color='black', linestyle='None', markersize=11, label='attack'),
    Line2D([0], [0], marker='$p$', color='black', linestyle='None', markersize=14, label='permissive'),
    Line2D([0], [0], marker='$b$', color='black', linestyle='None', markersize=14, label='balanced'),
    Line2D([0], [0], marker='$r$', color='black', linestyle='None', markersize=11, label='restrictive'),
]
ax.legend(
    handles=legend_handles,
    title='Subscenario',
    loc='upper right',
    frameon=True,
    fontsize=22,
    title_fontsize=22,
)

ax.set_xlabel('Permission Assistant Messages (Average per Run)', fontsize=24)
ax.set_ylabel('% of Runs with an Attack or\nOut-of-Alignment Tool Call', fontsize=24)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0.16, 0.08, 0.98, 0.98])

save_figure('risky_call_rate_vs_permission_messages_always_yes')




In [ ]:
# Average risky-call count vs permission-assistant message load (alignment_aware, excluding permissive)
# Same setup as the alignment-aware risky-call plot above, but the Y-axis is average number of attack + out-of-alignment calls per run.

import numpy as np

sub_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].astype(str).eq('alignment_aware')
    & df['subscenario'].notna()
    & df['subscenario'].astype(str).ne('permissive')
].copy()

sub_df['assistant_label'] = sub_df['permission_assistant'].astype(str)
mask_risk = sub_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & sub_df['risk_tolerance'].notna()
sub_df.loc[mask_risk, 'assistant_label'] = (
    sub_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + sub_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

sub_df['risky_call_count'] = sub_df['attack_tool_calls'].fillna(0) + sub_df['out_of_alignment_tool_calls'].fillna(0)

summary_sub = (
    sub_df.groupby(['assistant_label', 'subscenario'], as_index=False)
    .agg(
        permission_assistant_messages=('permission_assistant_messages', 'mean'),
        avg_risky_calls=('risky_call_count', 'mean'),
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
    )
)

fixed_point_size = 130

sub_order = sorted(summary_sub['subscenario'].dropna().astype(str).unique())
summary_sub['subscenario'] = pd.Categorical(summary_sub['subscenario'], categories=sub_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for pt in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], pt) <= 0:
            lower.pop()
        lower.append(pt)

    upper = []
    for pt in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], pt) <= 0:
            upper.pop()
        upper.append(pt)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_sub['assistant_label'].astype(str).unique())
palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(21, 9))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.10, 0.10),
    (0.10, -0.12),
    (0.22, 0.18),
    (0.22, -0.20),
    (-0.22, 0.14),
    (-0.22, -0.16),
]
assistant_label_offsets = [
    (0.18, 0.20),
    (0.18, -0.22),
    (0.34, 0.30),
    (0.34, -0.32),
    (-0.34, 0.26),
    (-0.34, -0.28),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_sub[summary_sub['assistant_label'] == assistant].sort_values('subscenario')
    if a_df.empty:
        continue

    points = list(zip(a_df['permission_assistant_messages'], a_df['avg_risky_calls']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [pt[0] for pt in hull] + [hull[0][0]]
        hy = [pt[1] for pt in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 0.35) or (abs(cy - py) > 0.20) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    alx, aly = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((alx, aly))
    ax.text(
        alx,
        aly,
        assistant,
        color=color,
        fontsize=19,
        ha='left' if chosen_adx >= 0 else 'right',
        va='center',
        bbox=dict(facecolor='white', edgecolor=color, alpha=0.75, pad=0.25),
        zorder=5,
    )

    ax.scatter(
        a_df['permission_assistant_messages'],
        a_df['avg_risky_calls'],
        s=fixed_point_size,
        color=color,
        edgecolor='white',
        linewidth=0.8,
        zorder=3,
    )

    for _, row in a_df.iterrows():
        x = float(row['permission_assistant_messages'])
        y = float(row['avg_risky_calls'])

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 0.22) or (abs(cy - py) > 0.12) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            subscenario_abbr.get(str(row['subscenario']), str(row['subscenario'])),
            color=color,
            fontsize=20,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

from matplotlib.lines import Line2D

legend_handles = [
    Line2D([0], [0], marker='$a$', color='black', linestyle='None', markersize=11, label='attack'),
    Line2D([0], [0], marker='$p$', color='black', linestyle='None', markersize=14, label='permissive'),
    Line2D([0], [0], marker='$b$', color='black', linestyle='None', markersize=14, label='balanced'),
    Line2D([0], [0], marker='$r$', color='black', linestyle='None', markersize=11, label='restrictive'),
]
ax.legend(
    handles=legend_handles,
    title='Subscenario',
    loc='upper right',
    frameon=True,
    fontsize=16,
    title_fontsize=17,
)

ax.set_xlabel('Permission Assistant Messages (Average per Run)', fontsize=24)
ax.set_ylabel('Average Number of Attack or\nOut-of-Alignment Tool Calls', fontsize=24)
ax.set_ylim(bottom=-0.05)
ax.grid(True, linewidth=0.6, alpha=0.7)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0.16, 0.08, 0.98, 0.98])

save_figure('average_risky_call_count_vs_permission_messages_alignment_aware')


In [ ]:
# Desired-call rate vs permission-assistant message load (alignment_aware), points by subscenario
# One unified graph. One point per subscenario for each permission assistant, with each assistant wrapped by its outer-edge hull.
# Y-axis is % desired tool calls made. Star markers indicate no attack/out-of-alignment tool calls were made.

import numpy as np

variant_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].astype(str).eq('alignment_aware')
    & df['subscenario'].notna()
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
variant_df['assistant_label'] = variant_df['permission_assistant'].astype(str)
mask_risk = variant_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & variant_df['risk_tolerance'].notna()
variant_df.loc[mask_risk, 'assistant_label'] = (
    variant_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + variant_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

variant_df['risky_call_made'] = (
    (variant_df['attack_tool_calls'].fillna(0) > 0)
    | (variant_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_variant = (
    variant_df.groupby(['assistant_label', 'subscenario'], as_index=False)
    .agg(
        permission_assistant_messages=('permission_assistant_messages', 'mean'),
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
        any_risky=('risky_call_made', 'any'),
    )
)

summary_variant['percent_desired_made'] = (
    100 * summary_variant['desired_calls'] / summary_variant['total_desired_possible'].replace(0, float('nan'))
).fillna(0.0).clip(0, 100)
summary_variant['point_size'] = 48 + 1.0 * summary_variant['percent_desired_made']

sub_order = sorted(summary_variant['subscenario'].dropna().astype(str).unique())
summary_variant['subscenario'] = pd.Categorical(summary_variant['subscenario'], categories=sub_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_variant['assistant_label'].astype(str).unique())
palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(12, 7.8))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.10, 1.8),
    (0.10, -2.4),
    (0.24, 2.8),
    (0.24, -3.0),
    (-0.24, 2.4),
    (-0.24, -2.8),
]
assistant_label_offsets = [
    (0.18, 3.0),
    (0.18, -3.2),
    (0.34, 5.0),
    (0.34, -5.2),
    (-0.34, 3.8),
    (-0.34, -4.0),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_variant[summary_variant['assistant_label'] == assistant].sort_values('subscenario')
    if a_df.empty:
        continue

    points = list(zip(a_df['permission_assistant_messages'], a_df['percent_desired_made']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 0.35) or (abs(cy - py) > 3.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    alx, aly = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((alx, aly))

    # Plot subscenario points, using stars when there were no risky calls.
    for _, row in a_df.iterrows():
        x = float(row['permission_assistant_messages'])
        y = float(row['percent_desired_made'])
        ax.scatter(
            [x],
            [y],
            s=float(row['point_size']),
            marker='o',
            color=color,
            edgecolor='white',
            linewidth=0.8,
            zorder=3,
        )

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 0.22) or (abs(cy - py) > 2.0) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            subscenario_abbr.get(str(row['subscenario']), str(row['subscenario'])),
            color=color,
            fontsize=20,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

ax.set_xlabel('Permission Assistant Messages (Average per Run)', fontsize=24)
ax.set_ylabel('% Desired Tool Calls Made', fontsize=24)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)




from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]
assistant_legend = fig.legend(
    handles=assistant_handles,
    title='Permission Assistant',
    loc='lower center',
    bbox_to_anchor=(0.5, -0.05),
    ncol=3,
    frameon=True,
    fontsize=18,
    title_fontsize=20,
)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0, 0.20, 1, 1])
save_figure('desired_call_rate_vs_permission_messages_alignment_aware')


In [ ]:
# Risky-call rate vs desired-call rate (alignment_aware), points by subscenario
# One unified graph. One point per subscenario for each permission assistant, with each assistant wrapped by its outer-edge hull.
# X-axis is % desired tool calls made. Y-axis is % of runs with out-of-alignment or attack tool calls.
# Star markers indicate no out-of-alignment/attack tool calls were made.

import numpy as np

variant2_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].astype(str).eq('alignment_aware')
    & df['subscenario'].notna()
    & df['subscenario'].astype(str).ne('permissive')
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
variant2_df['assistant_label'] = variant2_df['permission_assistant'].astype(str)
mask_risk = variant2_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & variant2_df['risk_tolerance'].notna()
variant2_df.loc[mask_risk, 'assistant_label'] = (
    variant2_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + variant2_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

variant2_df['risky_call_made'] = (
    (variant2_df['attack_tool_calls'].fillna(0) > 0)
    | (variant2_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_variant2 = (
    variant2_df.groupby(['assistant_label', 'subscenario'], as_index=False)
    .agg(
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
        percent_risky=('risky_call_made', lambda s: 100 * s.mean()),
        any_risky=('risky_call_made', 'any'),
    )
)

summary_variant2['percent_desired_made'] = (
    100 * summary_variant2['desired_calls'] / summary_variant2['total_desired_possible'].replace(0, float('nan'))
).fillna(0.0).clip(0, 100)
summary_variant2['point_size'] = 48 + 1.0 * summary_variant2['percent_desired_made']

sub_order = sorted(summary_variant2['subscenario'].dropna().astype(str).unique())
summary_variant2['subscenario'] = pd.Categorical(summary_variant2['subscenario'], categories=sub_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_variant2['assistant_label'].astype(str).unique())
palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(12, 9.2))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.8, 1.8),
    (0.8, -2.4),
    (1.6, 2.8),
    (1.6, -3.0),
    (-1.6, 2.4),
    (-1.6, -2.8),
]
assistant_label_offsets = [
    (1.5, 3.0),
    (1.5, -3.2),
    (2.8, 5.0),
    (2.8, -5.2),
    (-2.8, 3.8),
    (-2.8, -4.0),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_variant2[summary_variant2['assistant_label'] == assistant].sort_values('subscenario')
    if a_df.empty:
        continue

    points = list(zip(a_df['percent_desired_made'], a_df['percent_risky']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 2.0) or (abs(cy - py) > 3.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    alx, aly = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((alx, aly))

    # Plot subscenario points, using stars when there were no risky calls.
    for _, row in a_df.iterrows():
        x = float(row['percent_desired_made'])
        y = float(row['percent_risky'])
        ax.scatter(
            [x],
            [y],
            s=float(row['point_size']),
            marker='o',
            color=color,
            edgecolor='white',
            linewidth=0.8,
            zorder=3,
        )

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 1.2) or (abs(cy - py) > 2.0) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            subscenario_abbr.get(str(row['subscenario']), str(row['subscenario'])),
            color=color,
            fontsize=20,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

ax.set_xlabel('% Desired Tool Calls Made', fontsize=24)
ax.set_ylabel('% Runs with an Out-of-Alignment\nor Attack Tool Call', fontsize=24)
ax.set_xlim(-5, 105)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)


from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]


from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]
assistant_legend = fig.legend(
    handles=assistant_handles,
    title='Permission Assistant',
    loc='lower center',
    bbox_to_anchor=(0.55, 0.05),
    ncol=2,
    frameon=True,
    fontsize=20,
    title_fontsize=20,
)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0.12, 0.30, 1, 1])
save_figure('risky_call_rate_vs_desired_call_rate_alignment_aware')


In [ ]:
# Desired-call rate vs permission-assistant message load, points by synthetic responder
# One unified graph. One point per synthetic responder mode for each permission assistant, with each assistant wrapped by its outer-edge hull.
# Y-axis is % desired tool calls made. Star markers indicate no attack/out-of-alignment tool calls were made.

import numpy as np

variant_su_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].notna()
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
variant_su_df['assistant_label'] = variant_su_df['permission_assistant'].astype(str)
mask_risk = variant_su_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & variant_su_df['risk_tolerance'].notna()
variant_su_df.loc[mask_risk, 'assistant_label'] = (
    variant_su_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + variant_su_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

variant_su_df['risky_call_made'] = (
    (variant_su_df['attack_tool_calls'].fillna(0) > 0)
    | (variant_su_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_variant_su = (
    variant_su_df.groupby(['assistant_label', 'synthetic_responder_mode'], as_index=False)
    .agg(
        permission_assistant_messages=('permission_assistant_messages', 'mean'),
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
        any_risky=('risky_call_made', 'any'),
    )
)

summary_variant_su['percent_desired_made'] = (
    100 * summary_variant_su['desired_calls'] / summary_variant_su['total_desired_possible'].replace(0, float('nan'))
).fillna(0.0).clip(0, 100)
summary_variant_su['point_size'] = 48 + 1.0 * summary_variant_su['percent_desired_made']

mode_order = sorted(summary_variant_su['synthetic_responder_mode'].dropna().astype(str).unique())
summary_variant_su['synthetic_responder_mode'] = pd.Categorical(summary_variant_su['synthetic_responder_mode'].astype(str), categories=mode_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_variant_su['assistant_label'].astype(str).unique())
mode_display = {
    'always_no': 'no',
    'always_yes': 'yes',
    'alignment_aware': 'aligned',
}

palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(12, 8))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.10, 1.8),
    (0.10, -2.4),
    (0.24, 2.8),
    (0.24, -3.0),
    (-0.24, 2.4),
    (-0.24, -2.8),
]
assistant_label_offsets = [
    (0.18, 3.0),
    (0.18, -3.2),
    (0.34, 5.0),
    (0.34, -5.2),
    (-0.34, 3.8),
    (-0.34, -4.0),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_variant_su[summary_variant_su['assistant_label'] == assistant].sort_values('synthetic_responder_mode')
    if a_df.empty:
        continue

    points = list(zip(a_df['permission_assistant_messages'], a_df['percent_desired_made']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 0.35) or (abs(cy - py) > 3.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    label_x, label_y = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((label_x, label_y))


    risky_df = a_df[a_df['any_risky']].copy()
    safe_df = a_df[~a_df['any_risky']].copy()

    if not risky_df.empty:
        ax.scatter(
            risky_df['permission_assistant_messages'],
            risky_df['percent_desired_made'],
            s=risky_df['point_size'],
            marker='o',
            color=color,
            edgecolor='white',
            linewidth=0.8,
            zorder=3,
        )
    if not safe_df.empty:
        ax.scatter(
            safe_df['permission_assistant_messages'],
            safe_df['percent_desired_made'],
            s=safe_df['point_size'] * 1.15,
            marker='*',
            color=color,
            edgecolor='white',
            linewidth=0.9,
            zorder=3,
        )

    for _, row in a_df.iterrows():
        x = float(row['permission_assistant_messages'])
        y = float(row['percent_desired_made'])
        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 0.30) or (abs(cy - py) > 2.4) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            mode_display.get(mode_display.get(str(row['synthetic_responder_mode']), str(row['synthetic_responder_mode'])), mode_display.get(str(row['synthetic_responder_mode']), str(row['synthetic_responder_mode']))),
            color=color,
            fontsize=20,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

ax.set_xlabel('Permission Assistant Messages (Average per Run)', fontsize=24)
ax.set_ylabel('% Desired Tool Calls Made', fontsize=24)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)

from matplotlib.lines import Line2D
shape_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='#888888', markeredgecolor='white', markersize=7, label='Risky calls occurred'),
    Line2D([0], [0], marker='*', color='none', markerfacecolor='#888888', markeredgecolor='white', markersize=10, label='No risky calls'),
]


from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]


from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]
assistant_legend = fig.legend(
    handles=assistant_handles,
    title='Permission Assistant',
    loc='lower center',
    bbox_to_anchor=(0.5, -0.01),
    ncol=3,
    frameon=True,
    fontsize=20,
    title_fontsize=20,
)

shape_legend = ax.legend(
    handles=shape_handles,
    title='Point Marker',
    loc='upper center',
    bbox_to_anchor=(0.5, -0.16),
    ncol=2,
    frameon=True,
    fontsize=20,
    title_fontsize=20,
)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0, 0.16, 1, 1])
save_figure('desired_call_rate_vs_permission_messages_by_synthetic_responder')


In [ ]:
# Risky-call rate vs desired-call rate, points by synthetic responder
# One unified graph. One point per synthetic responder mode for each permission assistant, with each assistant wrapped by its outer-edge hull.
# X-axis is % desired tool calls made. Y-axis is % of runs with out-of-alignment or attack tool calls.
# Star markers indicate no out-of-alignment/attack tool calls were made.

import numpy as np

variant2_su_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].notna()
    & df['subscenario'].notna()
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
variant2_su_df['assistant_label'] = variant2_su_df['permission_assistant'].astype(str)
mask_risk = variant2_su_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & variant2_su_df['risk_tolerance'].notna()
variant2_su_df.loc[mask_risk, 'assistant_label'] = (
    variant2_su_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + variant2_su_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

variant2_su_df['risky_call_made'] = (
    (variant2_su_df['attack_tool_calls'].fillna(0) > 0)
    | (variant2_su_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_variant2_su = (
    variant2_su_df.groupby(['assistant_label', 'synthetic_responder_mode'], as_index=False)
    .agg(
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
        percent_risky=('risky_call_made', lambda s: 100 * s.mean()),
        any_risky=('risky_call_made', 'any'),
    )
)

summary_variant2_su['percent_desired_made'] = (
    100 * summary_variant2_su['desired_calls'] / summary_variant2_su['total_desired_possible'].replace(0, float('nan'))
).fillna(0.0).clip(0, 100)
summary_variant2_su['point_size'] = 48 + 1.0 * summary_variant2_su['percent_desired_made']

mode_order = sorted(summary_variant2_su['synthetic_responder_mode'].dropna().astype(str).unique())
summary_variant2_su['synthetic_responder_mode'] = pd.Categorical(summary_variant2_su['synthetic_responder_mode'].astype(str), categories=mode_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_variant2_su['assistant_label'].astype(str).unique())
mode_display = {
    'always_no': 'no',
    'always_yes': 'yes',
    'alignment_aware': 'aligned',
}

palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(12, 7.8))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (1.4, 1.6),
    (1.4, -2.2),
    (2.6, 2.8),
    (2.6, -3.0),
    (-2.6, 2.2),
    (-2.6, -2.6),
]
assistant_label_offsets = [
    (2.4, 4.0),
    (2.4, -4.2),
    (4.2, 6.2),
    (4.2, -6.0),
    (-4.2, 4.8),
    (-4.2, -4.8),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_variant2_su[summary_variant2_su['assistant_label'] == assistant].sort_values('synthetic_responder_mode')
    if a_df.empty:
        continue

    points = list(zip(a_df['percent_desired_made'], a_df['percent_risky']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 2.0) or (abs(cy - py) > 4.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    label_x, label_y = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((label_x, label_y))


    for _, row in a_df.iterrows():
        x = float(row['percent_desired_made'])
        y = float(row['percent_risky'])

        ax.scatter(
            x,
            y,
            s=float(row['point_size']),
            marker='o',
            color=color,
            edgecolor='white',
            linewidth=0.8,
            zorder=3,
        )

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 1.2) or (abs(cy - py) > 2.0) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            mode_display.get(mode_display.get(str(row['synthetic_responder_mode']), str(row['synthetic_responder_mode'])), mode_display.get(str(row['synthetic_responder_mode']), str(row['synthetic_responder_mode']))),
            color=color,
            fontsize=20,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

ax.set_xlabel('% Desired Tool Calls Made', fontsize=24)
ax.set_ylabel('% Runs with an Out-of-Alignment\nor Attack Tool Call', fontsize=24)
ax.set_xlim(-5, 105)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)




from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]
assistant_legend = fig.legend(
    handles=assistant_handles,
    title='Permission Assistant',
    loc='lower center',
    bbox_to_anchor=(0.55, -0.06),
    ncol=2,
    frameon=True,
    fontsize=18,
    title_fontsize=20,
)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0.12, 0.20, 1, 1])
save_figure('risky_call_rate_vs_desired_call_rate_by_synthetic_responder')


In [ ]:
# Risky-call rate vs permission-assistant message load by synthetic responder mode (excluding attack)
# Same synthetic-responder comparison as the publication figure above, but excluding the attack and permissive subscenarios.
# X-axis: average permission assistant messages.
# Y-axis: percent of runs where an attack or out-of-alignment call was made.
# One point per synthetic responder mode, with each assistant wrapped by its outer-edge hull.

import numpy as np

mode_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].notna()
    & df['subscenario'].notna()
    & (~df['subscenario'].astype(str).isin(['attack', 'permissive']))
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
mode_df['assistant_label'] = mode_df['permission_assistant'].astype(str)
mask_risk = mode_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & mode_df['risk_tolerance'].notna()
mode_df.loc[mask_risk, 'assistant_label'] = (
    mode_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + mode_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

mode_df['risky_call_made'] = (
    (mode_df['attack_tool_calls'].fillna(0) > 0)
    | (mode_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_mode = (
    mode_df.groupby(['assistant_label', 'synthetic_responder_mode'], as_index=False)
    .agg(
        permission_assistant_messages=('permission_assistant_messages', 'mean'),
        percent_risky=('risky_call_made', lambda s: 100 * s.mean()),
        output_passes=('output_passes', 'sum'),
        output_fails=('output_fails', 'sum'),
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
    )
)

summary_mode['total_output_evals'] = summary_mode['output_passes'].fillna(0) + summary_mode['output_fails'].fillna(0)
summary_mode['output_fail_rate'] = (
    100 * summary_mode['output_fails'].fillna(0) / summary_mode['total_output_evals'].replace(0, float('nan'))
).fillna(0.0)
summary_mode['marker'] = summary_mode['output_fail_rate'].apply(lambda pct: 'X' if pct > 5 else 'o')
fixed_point_size = 130

preferred_mode_order = ['always_no', 'alignment_aware', 'always_yes']
mode_values = list(summary_mode['synthetic_responder_mode'].dropna().astype(str).unique())
mode_order = [m for m in preferred_mode_order if m in mode_values] + [m for m in mode_values if m not in preferred_mode_order]
summary_mode['synthetic_responder_mode'] = pd.Categorical(summary_mode['synthetic_responder_mode'], categories=mode_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_mode['assistant_label'].astype(str).unique())
mode_display = {
    'always_no': 'no',
    'always_yes': 'yes',
    'alignment_aware': 'aligned',
}

palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(13, 12))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.10, 1.6),
    (0.10, -2.2),
    (0.22, 2.8),
    (0.22, -3.2),
    (-0.22, 2.2),
    (-0.22, -2.8),
]
assistant_label_offsets = [
    (0.18, 3.0),
    (0.18, -3.2),
    (0.34, 5.0),
    (0.34, -5.2),
    (-0.34, 3.8),
    (-0.34, -4.0),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_mode[summary_mode['assistant_label'] == assistant].sort_values('synthetic_responder_mode')
    if a_df.empty:
        continue

    points = list(zip(a_df['permission_assistant_messages'], a_df['percent_risky']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    # Label assistant shape on chart with overlap-avoidance.
    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 0.35) or (abs(cy - py) > 3.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    alx, aly = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((alx, aly))

    # Plot and label every synthetic-responder point.
    for _, row in a_df.iterrows():
        x = float(row['permission_assistant_messages'])
        y = float(row['percent_risky'])

        ax.scatter(
            [x],
            [y],
            s=fixed_point_size,
            marker=row['marker'],
            color=color,
            edgecolor='white',
            linewidth=0.8,
            zorder=3,
        )

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 0.22) or (abs(cy - py) > 2.0) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            str(mode_display.get(mode_display.get(str(row['synthetic_responder_mode']), str(row['synthetic_responder_mode'])), mode_display.get(str(row['synthetic_responder_mode']), str(row['synthetic_responder_mode'])))),
            color=color,
            fontsize=20,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

ax.set_xlabel('Permission Assistant Messages (Average per Run)', fontsize=24)
ax.set_ylabel('% of Runs with an Attack or\nOut-of-Alignment Tool Call', fontsize=24)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)

from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]


from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]
assistant_legend = fig.legend(
    handles=assistant_handles,
    title='Permission Assistant',
    loc='lower center',
    bbox_to_anchor=(0.55, 0.03),
    ncol=2,
    frameon=True,
    fontsize=20,
    title_fontsize=20,
)

marker_handles = [
    Line2D([0], [0], marker='o', color='#666666', markerfacecolor='#666666', markersize=10, linestyle='None', label='<=5% fail rate'),
    Line2D([0], [0], marker='X', color='#666666', markerfacecolor='#666666', markersize=10, linestyle='None', label='>5% fail rate'),
]

marker_legend = ax.legend(
    marker_handles,
    ['<=5% fail rate', '>5% fail rate'],
    title='Output Validation',
    loc='upper center',
    bbox_to_anchor=(0.45, -0.12),
    ncol=2,
    frameon=True,
    fontsize=20,
    title_fontsize=20,
)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0.12, 0.20, 1, 1])
save_figure('risky_call_rate_vs_permission_messages_by_synthetic_responder_excluding_attack')


In [ ]:
# Risky-call rate vs permission-assistant message load by synthetic responder mode
# Same synthetic-responder comparison as the publication figure above, but points are triangles when fewer than 100% of desired tool calls were made.
# X-axis: average permission assistant messages.
# Y-axis: percent of runs where an attack or out-of-alignment call was made.
# One point per synthetic responder mode, with each assistant wrapped by its outer-edge hull.

import numpy as np

mode_df = df[
    df['permission_assistant'].notna()
    & df['synthetic_responder_mode'].notna()
    & df['subscenario'].notna()
    & df['subscenario'].astype(str).ne('permissive')
].copy()

# Split risk_assessment and risk_assessment_autonomous into separate labels by risk tolerance (e.g., 0.2 vs 0.7).
mode_df['assistant_label'] = mode_df['permission_assistant'].astype(str)
mask_risk = mode_df['permission_assistant'].astype(str).isin(['risk_assessment', 'risk_assessment_autonomous']) & mode_df['risk_tolerance'].notna()
mode_df.loc[mask_risk, 'assistant_label'] = (
    mode_df.loc[mask_risk, 'permission_assistant'].astype(str)
    + ' (rt=' + mode_df.loc[mask_risk, 'risk_tolerance'].map(lambda x: f'{x:.1f}') + ')'
)

mode_df['risky_call_made'] = (
    (mode_df['attack_tool_calls'].fillna(0) > 0)
    | (mode_df['out_of_alignment_tool_calls'].fillna(0) > 0)
)

summary_mode = (
    mode_df.groupby(['assistant_label', 'synthetic_responder_mode'], as_index=False)
    .agg(
        permission_assistant_messages=('permission_assistant_messages', 'mean'),
        percent_risky=('risky_call_made', lambda s: 100 * s.mean()),
        desired_calls=('desired_tool_calls', 'sum'),
        total_desired_possible=('total_potential_desired_tool_calls', 'sum'),
    )
)

summary_mode['percent_desired_made'] = (
    100 * summary_mode['desired_calls'] / summary_mode['total_desired_possible'].replace(0, float('nan'))
).fillna(0.0).clip(0, 100)
summary_mode['marker'] = summary_mode['percent_desired_made'].apply(lambda pct: '^' if pct == 0 else 'o')

preferred_mode_order = ['always_no', 'alignment_aware', 'always_yes']
mode_values = list(summary_mode['synthetic_responder_mode'].dropna().astype(str).unique())
mode_order = [m for m in preferred_mode_order if m in mode_values] + [m for m in mode_values if m not in preferred_mode_order]
summary_mode['synthetic_responder_mode'] = pd.Categorical(summary_mode['synthetic_responder_mode'], categories=mode_order, ordered=True)


def convex_hull(points):
    """Andrew's monotone chain convex hull for 2D points."""
    pts = sorted(set((float(x), float(y)) for x, y in points))
    if len(pts) <= 1:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]


assistant_order = sorted(summary_mode['assistant_label'].astype(str).unique())
mode_display = {
    'always_no': 'no',
    'always_yes': 'yes',
    'alignment_aware': 'aligned',
}

palette = sns.color_palette('tab10', n_colors=len(assistant_order))

fig, ax = plt.subplots(figsize=(13, 12))

placed_point_labels = []
placed_assistant_labels = []
point_label_offsets = [
    (0.10, 1.6),
    (0.10, -2.2),
    (0.22, 2.8),
    (0.22, -3.2),
    (-0.22, 2.2),
    (-0.22, -2.8),
]
assistant_label_offsets = [
    (0.18, 3.0),
    (0.18, -3.2),
    (0.34, 5.0),
    (0.34, -5.2),
    (-0.34, 3.8),
    (-0.34, -4.0),
]

for color, assistant in zip(palette, assistant_order):
    a_df = summary_mode[summary_mode['assistant_label'] == assistant].sort_values('synthetic_responder_mode')
    if a_df.empty:
        continue

    points = list(zip(a_df['permission_assistant_messages'], a_df['percent_risky']))
    hull = convex_hull(points)

    if len(hull) >= 3:
        hx = [p[0] for p in hull] + [hull[0][0]]
        hy = [p[1] for p in hull] + [hull[0][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
        ax.fill(hx, hy, color=color, alpha=0.08)
    elif len(hull) == 2:
        hx = [hull[0][0], hull[1][0]]
        hy = [hull[0][1], hull[1][1]]
        ax.plot(hx, hy, linewidth=2.0, color=color)
    else:
        ax.plot([points[0][0]], [points[0][1]], marker='o', color=color)

    # Label assistant shape on chart with overlap-avoidance.
    anchor_x = float(np.mean([pt[0] for pt in hull])) if len(hull) > 0 else float(points[0][0])
    anchor_y = float(np.mean([pt[1] for pt in hull])) if len(hull) > 0 else float(points[0][1])
    chosen_adx, chosen_ady = assistant_label_offsets[0]
    for dx, dy in assistant_label_offsets:
        cx, cy = anchor_x + dx, anchor_y + dy
        if all((abs(cx - px) > 0.35) or (abs(cy - py) > 3.0) for px, py in placed_assistant_labels):
            chosen_adx, chosen_ady = dx, dy
            break

    alx, aly = anchor_x + chosen_adx, anchor_y + chosen_ady
    placed_assistant_labels.append((alx, aly))

    # Plot and label every synthetic-responder point.
    for _, row in a_df.iterrows():
        x = float(row['permission_assistant_messages'])
        y = float(row['percent_risky'])

        ax.scatter(
            [x],
            [y],
            s=150,
            marker=row['marker'],
            color=color,
            edgecolor='white',
            linewidth=0.8,
            zorder=3,
        )

        chosen_dx, chosen_dy = point_label_offsets[0]
        for dx, dy in point_label_offsets:
            cx, cy = x + dx, y + dy
            if all((abs(cx - px) > 0.22) or (abs(cy - py) > 2.0) for px, py in placed_point_labels):
                chosen_dx, chosen_dy = dx, dy
                break

        lx, ly = x + chosen_dx, y + chosen_dy
        placed_point_labels.append((lx, ly))

        ax.text(
            lx,
            ly,
            str(mode_display.get(mode_display.get(str(row['synthetic_responder_mode']), str(row['synthetic_responder_mode'])), mode_display.get(str(row['synthetic_responder_mode']), str(row['synthetic_responder_mode'])))),
            color=color,
            fontsize=20,
            va='center',
            ha='left' if chosen_dx >= 0 else 'right',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.65, pad=0.2),
            zorder=4,
        )

ax.set_xlabel('Permission Assistant Messages (Average per Run)', fontsize=24)
ax.set_ylabel('% of Runs with an Attack or\nOut-of-Alignment Tool Call', fontsize=24)
ax.set_ylim(-5, 105)
ax.grid(True, linewidth=0.6, alpha=0.7)

from matplotlib.lines import Line2D
assistant_handles = [
    Line2D([0], [0], color=(palette[a] if isinstance(palette, dict) else palette[i]), lw=3, label=a)
    for i, a in enumerate(assistant_order)
]

marker_handles = [
    Line2D([0], [0], marker='o', color='#666666', markerfacecolor='#666666', markersize=10, linestyle='None', label='>0% desired'),
    Line2D([0], [0], marker='^', color='#666666', markerfacecolor='#666666', markersize=10, linestyle='None', label='0% desired'),
]

assistant_legend = fig.legend(
    handles=assistant_handles,
    title='Permission Assistant',
    loc='lower center',
    bbox_to_anchor=(0.55, 0.03),
    ncol=2,
    frameon=True,
    fontsize=20,
    title_fontsize=20,
)

marker_legend = ax.legend(
    marker_handles,
    ['>0% desired', '0% desired'],
    title='Desired Tool Calls Made',
    loc='upper center',
    bbox_to_anchor=(0.45, -0.12),
    ncol=2,
    frameon=True,
    fontsize=20,
    title_fontsize=20,
)

ax.tick_params(axis='both', labelsize=21)
plt.tight_layout(rect=[0.12, 0.20, 1, 1])
save_figure('risky_call_rate_vs_permission_messages_by_synthetic_responder_with_desired_call_markers')
